# Residual Multitask Stellar Parameter Estimation

This notebook reproduces the workflow of the paper from SDSS DR12 spectra to the final estimates of `Teff`, `[Fe/H]`, and `log g`.

The sections follow the same logic as the paper: dataset loading, spectrum preprocessing, baseline comparison, hyperparameter search, final model training, and test-set evaluation.

In [ ]:
from pathlib import Path

import numpy as np

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
FIGS_DIR = PROJECT_ROOT / 'figs'
MODELS_DIR = PROJECT_ROOT / 'models'
RESULTS_DIR = PROJECT_ROOT / 'results_and_evaluations'

TRAIN_PARQUET = DATA_DIR / 'train.parquet'
VALIDATION_PARQUET = DATA_DIR / 'validation.parquet'
TEST_PARQUET = DATA_DIR / 'test.parquet'
BENCHMARK_NPZ = DATA_DIR / 'sdss_dr12_processed_flux_benchmark.npz'

for path in (DATA_DIR, FIGS_DIR, MODELS_DIR, RESULTS_DIR):
    path.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT.resolve()

## Data Status

Start by checking which local artifacts already exist. This cell reports the availability of the Parquet splits, the compact benchmark NPZ, the trained paper model, and the main evaluation outputs so you can decide which stages need to be rerun.

In [ ]:
def size_mb(path: Path) -> float:
    return path.stat().st_size / 1024**2 if path.exists() and path.is_file() else 0.0

status = {
    'train_parquet_exists': TRAIN_PARQUET.exists(),
    'train_parquet_mb': round(size_mb(TRAIN_PARQUET), 2),
    'validation_parquet_exists': VALIDATION_PARQUET.exists(),
    'validation_parquet_mb': round(size_mb(VALIDATION_PARQUET), 2),
    'test_parquet_exists': TEST_PARQUET.exists(),
    'test_parquet_mb': round(size_mb(TEST_PARQUET), 2),
    'benchmark_npz_exists': BENCHMARK_NPZ.exists(),
    'benchmark_npz_mb': round(size_mb(BENCHMARK_NPZ), 2),
}

status

## Data Acquisition

The paper uses a public SDSS DR12 benchmark with `50,000` spectra split into `30,000` training samples, `5,000` validation samples, and `15,000` test samples. This step downloads the Parquet splits from Hugging Face and makes the benchmark available locally.

In [ ]:
from data_acquisition.hf_data import ensure_all_parquet_splits

paths = ensure_all_parquet_splits(DATA_DIR)
paths

## Compact Dataset Construction

This step builds the compact NPZ used by the training and evaluation code. Following the paper, spectra are aligned to a common wavelength representation and the labels are prepared for multitask regression in a single file that is faster to load than the raw split files.

In [ ]:
from data_preprocessing.finalize_dataset import build_final_dataset

RUN_COMPACTIZATION = False
COMPACT_ROW_LIMIT = None
FORCE_COMPACTIZATION = False

if RUN_COMPACTIZATION:
    build_final_dataset(
        output_path=BENCHMARK_NPZ,
        row_limit=COMPACT_ROW_LIMIT,
        overwrite=FORCE_COMPACTIZATION,
    )
else:
    print('Set RUN_COMPACTIZATION = True to rebuild the compact NPZ.')

## Data Exploration

These optional plots summarize the benchmark used in the paper, including label coverage, sky distribution, SNR distribution, and examples of preprocessed spectra. They are useful for verifying that the train, validation, and test partitions remain consistent with the distributions discussed in the manuscript.

In [ ]:
from data_exploration import (
    plot_label_coverage,
    plot_preprocessed_spectrum,
    plot_sky_distribution,
    plot_snr_distribution,
    summarize_spectrum_uncertainties,
)

RUN_DATA_EXPLORATION = False
EXPLORATION_ROW_LIMIT_PER_SPLIT = None

if RUN_DATA_EXPLORATION:
    plot_label_coverage.main(row_limit_per_split=EXPLORATION_ROW_LIMIT_PER_SPLIT)
    plot_snr_distribution.main(row_limit_per_split=EXPLORATION_ROW_LIMIT_PER_SPLIT)
    plot_sky_distribution.main(row_limit_per_split=EXPLORATION_ROW_LIMIT_PER_SPLIT)
    plot_preprocessed_spectrum.main(row_limit=EXPLORATION_ROW_LIMIT_PER_SPLIT)
    summarize_spectrum_uncertainties.main(row_limit_per_split=EXPLORATION_ROW_LIMIT_PER_SPLIT)
else:
    print('Set RUN_DATA_EXPLORATION = True to generate exploration figures.')

## Classical Benchmarking (OLS and Ridge)

Before training the neural model, the paper compares it against linear baselines. This section runs `OLS` and `Ridge` on the same compact benchmark to show how much nonlinear modeling improves the estimation of `Teff`, `[Fe/H]`, and `log g`.

In [ ]:
from results_and_evaluations.model_benchmarks import print_benchmark_table, run_all_benchmarks
from sklearn.preprocessing import RobustScaler

RUN_BENCHMARKING = False

def target_scalers_from_compact(data):
    scalers = []
    for center, scale in zip(data['label_robust_center'], data['label_robust_scale']):
        scaler = RobustScaler()
        scaler.center_ = np.array([center])
        scaler.scale_ = np.array([scale])
        scaler.n_features_in_ = 1
        scalers.append(scaler)
    return scalers

if RUN_BENCHMARKING:
    data = np.load(BENCHMARK_NPZ, allow_pickle=True)
    benchmark_df = run_all_benchmarks(
        data['X_train_features'],
        data['y_train_targets'],
        data['X_val_features'],
        data['y_val_targets'],
        data['X_test_features'],
        data['y_test_targets'],
        target_scalers_from_compact(data),
        ['Teff', '[Fe/H]', 'log g'],
    )
    benchmark_csv = RESULTS_DIR / 'classical_benchmark_results.csv'
    benchmark_df.to_csv(benchmark_csv, index=False)
    print(f'Saved benchmark results to: {benchmark_csv}')
    print_benchmark_table(benchmark_df)
    benchmark_df.head()
else:
    print('Set RUN_BENCHMARKING = True to run the OLS and Ridge baselines.')

## Training And Evaluation Setup

This cell centralizes the paths and experiment constants used in the remaining sections. The defaults match the paper workflow: batch size `256`, augmentation factor `3`, Gaussian-noise level up to `0.05`, and the output paths for the saved model, metrics, scatter plot, and SNR analysis.

In [ ]:
BATCH_SIZE = 256
SEED = 42
AUG_FACTOR = 3
NOISE_LEVEL = 0.05
USE_XLA = False
PAPER_EPOCHS = 80

PAPER_MODEL_PATH = MODELS_DIR / 'paper_hyperparams_model.keras'
PAPER_WEIGHTS_PATH = MODELS_DIR / 'paper_hyperparams_best.weights.h5'
PAPER_METRICS_PATH = RESULTS_DIR / 'paper_hyperparams_metrics.json'
SCATTER_PNG_PATH = FIGS_DIR / 'paper_model_reference_vs_predicted.png'
SCATTER_METRICS_PATH = RESULTS_DIR / 'paper_model_test_metrics.json'
SNR_STATS_PATH = RESULTS_DIR / 'paper_model_snr_error_stats.csv'

print('Paper model path:', PAPER_MODEL_PATH)

## Hyperparameter Search

The paper selects the residual multitask MLP through Bayesian optimization over the stem width, shared residual trunk, task-specific heads, learning rate, weight decay, and dropout configuration.

Running this section performs the search, retrains the best configuration, writes tuner state under `results_and_evaluations/keras_tuner/`, and saves the searched model under `models/`.

In [ ]:
from argparse import Namespace

from data_preprocessing.augmentation import augment_training_data, prepare_multitask_dicts
from training.training import cast_target_dicts_to_float32, configure_cpu_runtime, plot_training_history, tune_and_train

RUN_HYPERPARAM_SEARCH = False
FORCE_HYPERPARAM_SEARCH = False
SEARCH_BATCH_SIZE = BATCH_SIZE
SEARCH_EPOCHS = 30
FINAL_TRAIN_EPOCHS = 30
TUNER_MAX_TRIALS = 100
TUNER_EXECUTIONS_PER_TRIAL = 1
SEARCH_ARCHITECTURE = 'baseline'
SEARCH_USE_XLA = False
SEARCH_FINE_TUNE_EPOCHS = 0
SEARCH_FINE_TUNE_LR = 1e-4

SEARCH_TUNER_DIRECTORY = RESULTS_DIR / 'keras_tuner'
SEARCH_TUNER_PROJECT_NAME = 'stellar_baseline_orchestration'
SEARCH_BEST_MODEL_PATH = MODELS_DIR / 'best_stellar_multitask_model.keras'
SEARCH_FINAL_MODEL_PATH = MODELS_DIR / 'stellar_regression_model.keras'

if RUN_HYPERPARAM_SEARCH:
    if SEARCH_FINAL_MODEL_PATH.exists() and not FORCE_HYPERPARAM_SEARCH:
        print(f'Searched final model already exists: {SEARCH_FINAL_MODEL_PATH.resolve()}')
        print('Set FORCE_HYPERPARAM_SEARCH = True to rerun the tuner and final training.')
    else:
        runtime_info = configure_cpu_runtime()
        print('CPU runtime configured:', runtime_info)

        data = np.load(BENCHMARK_NPZ, allow_pickle=True)
        x_train = data['X_train_features']
        y_train = data['y_train_targets'].astype(np.float32)
        x_val = data['X_val_features']
        y_val = data['y_val_targets'].astype(np.float32)

        y_train_dict, y_val_dict = prepare_multitask_dicts(y_train, y_val)
        y_train_dict, y_val_dict = cast_target_dicts_to_float32(y_train_dict, y_val_dict)

        if AUG_FACTOR > 1:
            x_train, y_train_dict = augment_training_data(
                x_train,
                y_train_dict,
                aug_factor=AUG_FACTOR,
                noise_level=NOISE_LEVEL,
                seed=SEED,
            )

        tuner_outputs = tune_and_train(
            x_train,
            y_train_dict,
            x_val,
            y_val_dict,
            tuner_directory=str(SEARCH_TUNER_DIRECTORY),
            tuner_project_name=SEARCH_TUNER_PROJECT_NAME,
            model_checkpoint_path=str(SEARCH_BEST_MODEL_PATH),
            final_model_path=str(SEARCH_FINAL_MODEL_PATH),
            architecture=SEARCH_ARCHITECTURE,
            tuner_max_trials=TUNER_MAX_TRIALS,
            tuner_executions_per_trial=TUNER_EXECUTIONS_PER_TRIAL,
            search_epochs=SEARCH_EPOCHS,
            search_batch_size=SEARCH_BATCH_SIZE,
            final_train_epochs=FINAL_TRAIN_EPOCHS,
            fine_tune_epochs=SEARCH_FINE_TUNE_EPOCHS,
            fine_tune_learning_rate=SEARCH_FINE_TUNE_LR,
            jit_compile=SEARCH_USE_XLA,
        )
        plot_training_history(tuner_outputs['history'])
        tuner_outputs
else:
    print('Set RUN_HYPERPARAM_SEARCH = True to run Bayesian hyperparameter search plus final retraining.')

## Train The Paper Model

This section trains the fixed architecture selected in the paper. The saved configuration has `542,771` trainable parameters, uses one `64`-unit residual block after a `128`-unit stem, and applies multitask heads specialized for `Teff`, `[Fe/H]`, and `log g`.

In [ ]:
from argparse import Namespace

from training.train_paper_hyperparams import train_paper_model

RUN_PAPER_TRAINING = False
USE_CPU = False
NO_MIXED_PRECISION = False
CACHE_DATASETS = False

paper_args = Namespace(
    data=BENCHMARK_NPZ,
    model=PAPER_MODEL_PATH,
    weights=PAPER_WEIGHTS_PATH,
    results=PAPER_METRICS_PATH,
    epochs=PAPER_EPOCHS,
    batch_size=BATCH_SIZE,
    aug_factor=AUG_FACTOR,
    noise_level=NOISE_LEVEL,
    seed=SEED,
    use_xla=USE_XLA,
    cpu=USE_CPU,
    no_mixed_precision=NO_MIXED_PRECISION,
    cache=CACHE_DATASETS,
)

if RUN_PAPER_TRAINING:
    paper_outputs = train_paper_model(paper_args)
    paper_outputs
else:
    print('Set RUN_PAPER_TRAINING = True to train the paper-selected model.')

## Generate Scatter Plot And Test Metrics

After training, this step runs inference on the held-out test set and generates the paper-style predicted-versus-reference scatter plot. It also saves the main metrics JSON, with test MAE values close to `59.5 K` for `Teff`, `0.101 dex` for `[Fe/H]`, and `0.130 dex` for `log g` in this checkout.

In [ ]:
from argparse import Namespace

from results_and_evaluations.infer_and_plot_scatter import run_inference_and_plot

RUN_SCATTER_EVALUATION = False

scatter_args = Namespace(
    model=PAPER_MODEL_PATH,
    data=BENCHMARK_NPZ,
    prediction_scalers=None,
    out=SCATTER_PNG_PATH,
    metrics=SCATTER_METRICS_PATH,
    batch_size=1024,
    dpi=300,
    base_font=14,
    annotation_font=13,
    point_size=1.3,
    alpha=0.78,
    figsize=(8.0, 2.2),
)

if RUN_SCATTER_EVALUATION:
    scatter_metrics = run_inference_and_plot(scatter_args)
    scatter_metrics
else:
    print('Set RUN_SCATTER_EVALUATION = True to save the scatter plot.')

## Generate SNR-Binned Error Statistics

The final evaluation breaks the prediction errors down by signal-to-noise ratio. This mirrors the paper's interest in realistic performance across spectra of different quality instead of relying only on aggregate test-set metrics.

In [ ]:
from argparse import Namespace

from results_and_evaluations.calculate_snr_error_std import calculate_stats

RUN_SNR_EVALUATION = False

snr_args = Namespace(
    model=PAPER_MODEL_PATH,
    data=BENCHMARK_NPZ,
    snr_data=TEST_PARQUET,
    prediction_scalers=None,
    out=SNR_STATS_PATH,
    batch_size=1024,
    snr_min=0,
    snr_max=140,
    bin_width=20,
    ddof=0,
)

if RUN_SNR_EVALUATION:
    snr_stats = calculate_stats(snr_args)
    snr_stats
else:
    print('Set RUN_SNR_EVALUATION = True to compute SNR-binned errors.')